In [ ]:
# ============================================================
# PropInsight Directory Configuration (Colab, Final Version)
# ============================================================
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/PropInsight"
os.makedirs(BASE, exist_ok=True)
os.makedirs(f"{BASE}/data", exist_ok=True)
os.makedirs(f"{BASE}/models", exist_ok=True)
os.makedirs(f"{BASE}/results", exist_ok=True)
os.makedirs(f"{BASE}/visualizations", exist_ok=True)

print(f"✅ Base directory: {BASE}")
print(f"✅ All imports complete!")


# --- Root ---
BASE = Path("/content/drive/MyDrive/PropInsight")

# --- Primary Folders ---
CORPUS_DIR      = BASE / "corpus"
LABELED_DIR     = BASE / "labeled"
RAW_DIR         = BASE / "raw"
RESI_DIR        = BASE / "RESI"
PROCESSED_DIR   = BASE / "processed"
PREPROCESS_DIR  = BASE / "preprocess"
ANALYTICS_DIR   = BASE / "analytics"
CACHE_DIR       = BASE / "cache"

# --- Labeled Training Datasets ---
SGEXPATS_FORUM  = LABELED_DIR / "forums/singapore_property_forum_posts_sgexpats_processed/sgexpats_forum_labeled.csv"
HWZ_FORUM       = LABELED_DIR / "multi_forum_property_posts_hwz_processed_2023_2025/forum_labeled_corrected.csv"
GOV_WEBSITES    = LABELED_DIR / "government/gov_websites_labeled.csv"
REDDIT_PROPERTY = LABELED_DIR / "reddit/reddit_2023_2025_property_labeled.csv"

# --- Corpus & Reference Data ---
SINGLISH_DIR  = CORPUS_DIR / "Singlish"          # dictionary/, vocabulary/, lexicon.csv
PROP_DIR      = CORPUS_DIR / "SGPropertyDomain"  # glossary.csv, regex_patterns.jsonl, etc.

# --- Generated Training Files ---
DATASETS_DIR   = BASE / "propinsight_datasets"
TRAIN_JSON     = DATASETS_DIR / "train.json"
VAL_JSON       = DATASETS_DIR / "validation.json"
TEST_JSON      = DATASETS_DIR / "test.json"
META_JSON      = DATASETS_DIR / "dataset_metadata.json"
LABEL_MAPS     = DATASETS_DIR / "label_maps.json"
CLASS_WEIGHTS  = DATASETS_DIR / "class_weights.json"

# --- Model + Results + Visualizations ---
MODELS_DIR         = BASE / "models"
FINETUNED_DIR      = MODELS_DIR / "finetuned"
RESULTS_DIR        = BASE / "results"
VISUALIZATIONS_DIR = BASE / "visualizations"

# --- Output & Results Files ---
TRAINING_CONFIG     = RESULTS_DIR / "training_config.json"
EVAL_RESULTS        = RESULTS_DIR / "evaluation_results.json"
TRAINING_CURVES     = RESULTS_DIR / "training_curves.png"
EVAL_PLOTS          = RESULTS_DIR / "evaluation_plots.png"
EVAL_REPORT_TXT     = RESULTS_DIR / "evaluation_report.txt"
BASELINE_PREDICTIONS  = RESULTS_DIR / "baseline_predictions.json"
FINETUNED_PREDICTIONS = RESULTS_DIR / "finetuned_predictions.json"
COMPREHENSIVE_METRICS = RESULTS_DIR / "comprehensive_metrics.json"
EVAL_REPORT_JSON      = RESULTS_DIR / "evaluation_report.json"
RESI_ALIGNMENT_JSON   = RESULTS_DIR / "resi_alignment.json"

# --- Logging & Monitoring ---
TENSORBOARD_LOGS = RESULTS_DIR / "tensorboard"
TRAINING_LOG     = RESULTS_DIR / "qwen_sealion_finetune.log"

# --- RESI Evaluation ---
RESI_BENCHMARK  = RESI_DIR
RESI_ALIGNMENT  = RESI_ALIGNMENT_JSON

# --- Evaluation Outputs (standalone folder) ---
EVAL_OUT_DIR            = BASE / "evaluation_results"
STANDALONE_EVAL_RESULTS = EVAL_OUT_DIR / "evaluation_results.json"

# --- Ensure all directories exist ---
for d in [
    BASE, CORPUS_DIR, LABELED_DIR, RAW_DIR, RESI_DIR, PROCESSED_DIR,
    PREPROCESS_DIR, ANALYTICS_DIR, CACHE_DIR, DATASETS_DIR, MODELS_DIR,
    FINETUNED_DIR, RESULTS_DIR, VISUALIZATIONS_DIR, TENSORBOARD_LOGS,
    EVAL_OUT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Directory configuration complete.")
print(f"Base: {BASE}")

In [ ]:
# ============================================================
# PropInsight Analytics Post-Processing (Baseline + Finetuned)
# Builds: Sentiment Index, Topic–Sentiment Matrix, Community
# Comparison, Emotion distribution, Policy correlation deltas.
# Saves under results/analytics/{baseline,finetuned}
# ============================================================

import json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# ---- Paths from your config cell ----
RESULTS_DIR = OUTPUT_DIR
BASELINE_PATH   = RESULTS_DIR / "baseline_predictions.json"
FINETUNED_PATH  = RESULTS_DIR / "finetuned_predictions.json"
ANALYTICS_DIR   = RESULTS_DIR / "analytics"
VIZ_DIR         = BASE / "visualizations"

ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)
(VIZ_DIR / "baseline").mkdir(parents=True, exist_ok=True)
(VIZ_DIR / "finetuned").mkdir(parents=True, exist_ok=True)

# ---- Helpers ----
FIELD_KEYS = [
  ("Overall Sentiment","overall_sentiment"),
  ("Price Sentiment","price_sentiment"),
  ("Policy Sentiment","policy_sentiment"),
  ("Affordability Sentiment","affordability_sentiment"),
  ("Location","location"),
  ("Aspect","aspect"),
  ("Entity","entity"),
  ("Policy Mentioned","policy_mentioned"),
  ("Singlish Detected","singlish_detected"),
  ("Cultural Context","cultural_context"),
  ("Emotion","emotion"),
  ("Datetime","datetime"),
  ("Source","source"),
  ("Reasoning","reasoning"),
]

SENTI_SCORE = {"optimistic": +1.0, "positive": +0.8, "neutral": 0.0, "pessimistic": -1.0, "negative": -0.8}

def parse_block(text: str) -> dict:
    out = {dst: "" for _, dst in FIELD_KEYS}
    for src, dst in FIELD_KEYS:
        m = re.search(rf"{re.escape(src)}\s*:\s*(.*)", text or "", flags=re.IGNORECASE)
        if m: out[dst] = m.group(1).strip()
    # normalize
    out["singlish_detected"] = str(out.get("singlish_detected","")).lower() in {"yes","true","1"}
    for k in ["overall_sentiment","price_sentiment","policy_sentiment","affordability_sentiment","emotion"]:
        out[k] = str(out.get(k,"")).strip().lower() or "neutral"
    if not out.get("location"): out["location"] = "Unknown"
    return out

def build_df(preds_json: Path) -> pd.DataFrame:
    data = json.load(open(preds_json, "r", encoding="utf-8"))
    rows = []
    for p in data:
        md = p.get("metadata", {}) or {}
        parsed = parse_block(p.get("prediction",""))
        # backfill from metadata if missing
        for k in ["datetime","source","location","aspect","entity","emotion"]:
            if not parsed.get(k): parsed[k] = str(md.get(k,""))
        # engagement for weighting
        eng = md.get("engagement_score", 0)
        try:
            eng = float(eng)
        except:
            eng = 0.0
        parsed["engagement_score"] = max(eng, 0.0)
        rows.append(parsed)
    df = pd.DataFrame(rows)
    # types
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    return df

def build_sentiment_index(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["senti_score"] = d["overall_sentiment"].map(lambda x: SENTI_SCORE.get(str(x).lower(), 0.0))
    w = d.get("engagement_score", pd.Series([1]*len(d)))
    w = pd.to_numeric(w, errors="coerce").fillna(1).clip(lower=1)
    d["wscore"] = d["senti_score"] * w
    # per-day per-location
    daily = (
        d.groupby([pd.Grouper(key="datetime", freq="D"), "location"], dropna=False)
         .apply(lambda g: g["wscore"].sum() / (pd.to_numeric(g["engagement_score"], errors="coerce").fillna(1).clip(lower=1).sum()))
         .rename("sentiment_index")
         .reset_index()
    )
    return daily

def topic_sentiment_matrix(df: pd.DataFrame) -> pd.DataFrame:
    mat = (df.groupby(["aspect","overall_sentiment"]).size().unstack(fill_value=0))
    mat["total"] = mat.sum(axis=1)
    return mat

def community_comparison(df: pd.DataFrame) -> pd.DataFrame:
    # Distribution of sentiments by source/community
    return (df.groupby(["source","overall_sentiment"]).size()
              .unstack(fill_value=0)
              .sort_index())

def emotion_distribution(df: pd.DataFrame) -> pd.DataFrame:
    vc = df["emotion"].fillna("neutral").str.lower().value_counts()
    return vc.rename_axis("emotion").reset_index(name="count")

def correlate_policies(daily_idx: pd.DataFrame, policy_events: pd.DataFrame, window=14) -> pd.DataFrame:
    # average across locations first
    g = (daily_idx.groupby("datetime", as_index=False)["sentiment_index"].mean()
                     .sort_values("datetime").dropna())
    def delta_at(date):
        t0 = pd.to_datetime(date)
        pre  = g[(g["datetime"]>=t0-pd.Timedelta(days=window)) & (g["datetime"]<t0)]["sentiment_index"].mean()
        post = g[(g["datetime"]> t0) & (g["datetime"]<=t0+pd.Timedelta(days=window))]["sentiment_index"].mean()
        return float(post - pre)
    out = policy_events.copy()
    out["delta_index"] = out["date"].apply(delta_at)
    return out.sort_values("date")

def run_all(preds_path: Path, tag: str):
    print(f"▶ Processing {tag}: {preds_path}")
    if not preds_path.exists():
        print(f"… skipped, file not found.")
        return
    df = build_df(preds_path)
    # Save parsed rows for audit
    out_dir = ANALYTICS_DIR / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / "parsed_predictions.csv", index=False)

    # Core analytics
    daily_idx = build_sentiment_index(df)
    daily_idx.to_csv(out_dir / "sentiment_index_daily.csv", index=False)

    tsm = topic_sentiment_matrix(df)
    tsm.to_csv(out_dir / "topic_sentiment_matrix.csv")

    comm = community_comparison(df)
    comm.to_csv(out_dir / "community_comparison.csv")

    emo = emotion_distribution(df)
    emo.to_csv(out_dir / "emotion_distribution.csv", index=False)

    # Example policy events (replace with your real list)
    policy_events = pd.DataFrame([
        {"event":"ABSD hike",    "date":"2023-04-27"},
        {"event":"BTO Dec 2023", "date":"2023-12-05"},
        # add more…
    ])
    corr = correlate_policies(daily_idx, policy_events, window=14)
    corr.to_csv(out_dir / "policy_correlation.csv", index=False)

    # Quick plots
    # Sentiment Index (averaged)
    avg = daily_idx.groupby("datetime", as_index=False)["sentiment_index"].mean()
    plt.figure(figsize=(12,4))
    plt.plot(avg["datetime"], avg["sentiment_index"])
    plt.title(f"Sentiment Index (avg across locations) — {tag}")
    plt.ylabel("Index"); plt.xlabel("Date"); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(VIZ_DIR / tag / "sentiment_index.png", dpi=300); plt.close()

    # Topic–Sentiment bar (top 10 aspects by total)
    tsm2 = tsm.sort_values("total", ascending=False).head(10).drop(columns=["total"], errors="ignore")
    tsm2.plot(kind="bar", stacked=True, figsize=(12,5), title=f"Topic–Sentiment Matrix (Top 10) — {tag}")
    plt.tight_layout(); plt.savefig(VIZ_DIR / tag / "topic_sentiment_matrix.png", dpi=300); plt.close()

    print(f"✓ Saved analytics to {out_dir}")
    print(f"✓ Plots    to {VIZ_DIR / tag}")

# ----- Run for both baseline and finetuned -----
run_all(BASELINE_PATH,  "baseline")
run_all(FINETUNED_PATH, "finetuned")
print("✅ Analytics complete")
